In [ ]:
import numpy as np
import numpy.typing as npt
from sklearn import datasets
import typing
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

digits = datasets.load_digits()
first_digit = digits.data[0]
first_digit

In [ ]:
target0 = digits.target[0]
print(f"Class of the first image: '{target0}'")

In [ ]:
print(f"Shape of digits: '{digits.data.shape}'")
print(f"Shape of an element: '{digits.data[0].shape}'")

In [ ]:
X = digits.data.reshape(-1, 1, 64)
print(f"Shape of X: '{X.shape}'")
print(f"Shape of an element: '{X[0].shape}'")

In [ ]:
y = digits.target
print(f"Shape of y: '{y.shape}'")
print("First ten values of y:")
print(y[:10])

In [ ]:
from sklearn.preprocessing import OneHotEncoder
enc = OneHotEncoder()
y_reshaped = enc.fit_transform(y.reshape(-1, 1)).toarray()
print("first element of y_reshaped")
print(y_reshaped[0])

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y_reshaped, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

Training set size: 1437
Testing set size: 360


In [8]:
def sigmoid(x: npt.NDArray[np.float64]) -> npt.NDArray[np.float64]:
    """Computes the sigmoid of x (our activation function)."""
    return 1.0 / (1.0 + np.exp(-x))

In [9]:
def sigmoid_prime(x:npt.NDArray[np.float64]) -> npt.NDArray[np.float64]:
    """TODO!"""
    pass

In [10]:
def mse_prime(target: npt.NDArray[np.float64], predicted:npt.NDArray[np.float64]) -> npt.NDArray[np.float64]:
    """TODO!"""
    pass

In [13]:
class FullyConnectedLayer:
    weights: npt.NDArray[np.float64]
    bias: npt.NDArray[np.float64]
    delta_w: npt.NDArray[np.float64]
    delta_b: npt.NDArray[np.float64]
    passes: int
    input: typing.Optional[npt.NDArray[np.float64]]

    def __init__(self, input_size: int, output_size: int):
        self.weights = np.random.randn(input_size, output_size) / np.sqrt(input_size)
        self.bias = np.zeros((1, output_size))
        self.delta_w = np.zeros_like(self.weights)
        self.delta_b = np.zeros_like(self.bias)
        self.passes = 0 

    def forward(self, input_data: npt.NDArray[np.float64]) -> npt.NDArray[np.float64]:
        self.input = input_data
        return np.dot(self.input, self.weights) + self.bias

    def backward(self, output_error: npt.NDArray[np.float64]) -> npt.NDArray[np.float64]:
        input_error = np.dot(output_error, self.weights.T)
        weights_error = np.dot(self.input.T, output_error)

        #  accumulate the gradient over the minibatch
        self.delta_w += weights_error
        self.delta_b += output_error
        self.passes += 1
        return input_error

    def step(self, eta: float) -> None:
        self.weights -= eta * self.delta_w / self.passes
        self.bias -= eta * self.delta_b / self.passes
        #  reset for the next minibatch
        self.delta_w = np.zeros(self.weights.shape)
        self.delta_b = np.zeros(self.bias.shape)
        self.passes = 0

In [14]:
class ActivationLayer:
    input: typing.Optional[npt.NDArray[np.float64]]

    def forward(self, input_data: npt.NDArray[np.float64]) -> npt.NDArray[np.float64]:
        self.input = input_data
        return sigmoid(input_data)

    def backward(self, output_error: npt.NDArray[np.float64]) -> npt.NDArray[np.float64]:
        return sigmoid_prime(self.input) * output_error
    
    def step(self, eta: float) -> None:
        return

In [15]:
Layer = typing.Union[FullyConnectedLayer, ActivationLayer]

class Network:
    def __init__(self, verbose=True):
        self.verbose = verbose
        self.layers = []

    def add(self, layer: Layer) -> None:
        self.layers.append(layer)

    def predict(self, input_data: npt.NDArray[np.float64]) -> typing.List[npt.NDArray[np.float64]]:
        result = []
        for i in range(input_data.shape[0]):
            output = input_data[i]
            for layer in self.layers:
                output = layer.forward(output)
            result.append(output)
        return result
        
    def fit(self,  x_train: npt.NDArray[np.float64], y_train: npt.NDArray[np.float64], 
            minibatches: int,  learning_rate: float, batch_size: int=64) -> None:
        for i in range(minibatches):
            idx = np.argsort(np.random.random(x_train.shape[0]))[:batch_size]
            x_batch = x_train[idx]
            y_batch = y_train[idx]
            for j in range(batch_size):
                output = x_batch[j]
                for layer in self.layers:
                    output = layer.forward(output)
                error = mse_prime(y_batch[j], output)
                for layer in reversed(self.layers):
                    error = layer.backward(error)
            for layer in self.layers:
                layer.step(learning_rate)

In [16]:
network = Network()
network.add(FullyConnectedLayer(64, 128))
network.add(ActivationLayer())
network.add(FullyConnectedLayer(128, 64))
network.add(ActivationLayer())
network.add(FullyConnectedLayer(64, 10))
network.add(ActivationLayer())

network.fit(X_train, y_train, minibatches=10000, learning_rate=0.5)

In [ ]:
xt = X_test[0]
yt = y_test[0]
out = network.predict(xt)
print(out)
print(yt)

In [ ]:
def highest_probability_digit(out: typing.List[npt.NDArray[np.float64]]) -> int:
    """Return the digit (0-9) the network assigns the highest probability to. TODO!"""
    pass

In [ ]:
# fill in highest_probability_digit so that this is true
highest_probability_digit(out) == np.argmax(y_test[0])

In [ ]:
def test_accuracy(X_test, y_test) -> float:
    """Write a function which returns the test accuracy 
    (AKA how often the prediction is correct on the test data X_test). 
    It should be a float between 0 and 1. TODO!"""
    pass

In [ ]:
# What is the test accuracy after 1000 minibatches?
# What is the test accuracy after 2000 minibatches?